<a href="https://colab.research.google.com/github/HowardChen0211/dataManPy/blob/main/ProblemSet_1_pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction

## U.S. Housing Market Trends - California

### Research Question

- How have home values, rental prices changed in California, and to what extent is home-value growth associated with rent growth?

- Which counties have seen the fastest rising housing prices?


### Unit of Analysis

Housing price analysis for 58 counties; rental price analysis for 49 counties.


### Objectives

1. Clean and standardize Zillow housing datasets.
2. Transform monthly housing data into an analysis-ready format.
3. Integrate home value, rental, and inventory datasets.
4. Analyze housing trends across metropolitan areas.
5. Examine the relationship between home value and rent growth.

# ProblemSet 0

## Import Libraries

In [66]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

#will display all output not just last command
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

pd.set_option('display.float_format', '{:.3f}'.format) # Turn off the scientific notation https://medium.com/@amit25173/steps-to-pandas-turn-off-scientific-notation-62c44b86ee24

## Load the data

In [67]:
# Zillow Home Value Index (ZHVI)
!wget -q -O ZHVI.csv https://files.zillowstatic.com/research/public_csvs/zhvi/County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv
# Zillow Observed Rent Index (ZORI)
!wget -q -O ZORI.csv https://files.zillowstatic.com/research/public_csvs/zori/County_zori_uc_sfrcondomfr_sm_month.csv

In [68]:
!ls

sample_data  ZHVI.csv  ZORI.csv


In [69]:
# Read the data
zhvi = pd.read_csv("ZHVI.csv")
zori = pd.read_csv("ZORI.csv")

In [70]:
# See how many features in the data
print("Home Index: ", zhvi.columns)
print("Rent Index: ", zori.columns)

Home Index:  Index(['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName',
       'State', 'Metro', 'StateCodeFIPS', 'MunicipalCodeFIPS', '2000-01-31',
       ...
       '2025-11-30', '2025-12-31', '2026-01-31', '2026-02-28', '2026-03-31',
       '2026-04-30', '2026-05-31', '2026-06-30', '2026-07-31', '2026-08-31'],
      dtype='object', length=329)
Rent Index:  Index(['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName',
       'State', 'Metro', 'StateCodeFIPS', 'MunicipalCodeFIPS', '2015-01-31',
       ...
       '2025-11-30', '2025-12-31', '2026-01-31', '2026-02-28', '2026-03-31',
       '2026-04-30', '2026-05-31', '2026-06-30', '2026-07-31', '2026-08-31'],
      dtype='object', length=149)


In [71]:
# take a look about the data (first 5 rows)
zhvi.head()

,RegionID,SizeRank,RegionName,RegionType,StateName,State,Metro,StateCodeFIPS,MunicipalCodeFIPS,2000-01-31,...,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31,2026-04-30,2026-05-31,2026-06-30,2026-07-31,2026-08-31
0,3101,0,Los Angeles County,county,CA,CA,"Los Angeles-Long Beach-Anaheim, CA",6,37,210217.606,...,876483.587,880007.554,881688.904,881664.430,879853.110,877155.644,874790.007,872663.237,872220.754,873008.422
1,139,1,Cook County,county,IL,IL,"Chicago-Naperville-Elgin, IL-IN-WI",17,31,152726.258,...,327258.568,328895.004,330363.857,332095.628,333712.156,335053.112,336017.428,337263.419,339162.650,341569.751
2,1090,2,Harris County,county,TX,TX,"Houston-The Woodlands-Sugar Land, TX",48,201,113021.492,...,283353.950,283400.843,283252.913,282917.698,282310.099,281596.697,280609.690,279554.676,278797.692,278517.967
3,2402,3,Maricopa County,county,AZ,AZ,"Phoenix-Mesa-Chandler, AZ",4,13,146106.504,...,459582.164,460367.020,461220.421,461894.031,461740.422,460735.658,459147.558,457589.894,456733.110,456252.051
4,2841,4,San Diego County,county,CA,CA,"San Diego-Chula Vista-Carlsbad, CA",6,73,216416.160,...,925424.341,927598.249,928828.819,929848.898,930228.330,930115.693,929543.266,928860.114,929330.518,930799.655


In [72]:
# take a look about the data (first 5 rows)
zori.head()

,RegionID,SizeRank,RegionName,RegionType,StateName,State,Metro,StateCodeFIPS,MunicipalCodeFIPS,2015-01-31,...,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31,2026-04-30,2026-05-31,2026-06-30,2026-07-31,2026-08-31
0,3101,0,Los Angeles County,county,CA,CA,"Los Angeles-Long Beach-Anaheim, CA",6,37,1691.320,...,2777.262,2768.420,2770.406,2776.600,2788.838,2797.135,2805.344,2809.822,2810.309,2813.614
1,139,1,Cook County,county,IL,IL,"Chicago-Naperville-Elgin, IL-IN-WI",17,31,1413.147,...,2145.212,2143.577,2157.834,2178.380,2204.530,2226.641,2248.019,2265.658,2274.611,2272.117
2,1090,2,Harris County,county,TX,TX,"Houston-The Woodlands-Sugar Land, TX",48,201,1171.144,...,1577.797,1572.948,1571.596,1569.418,1570.920,1573.802,1580.332,1584.251,1586.397,1586.095
3,2402,3,Maricopa County,county,AZ,AZ,"Phoenix-Mesa-Chandler, AZ",4,13,905.494,...,1692.043,1689.654,1690.670,1697.466,1702.306,1706.189,1711.234,1712.111,1715.836,1716.663
4,2841,4,San Diego County,county,CA,CA,"San Diego-Chula Vista-Carlsbad, CA",6,73,1628.319,...,2917.657,2912.131,2915.513,2929.541,2941.689,2951.787,2963.444,2976.321,2984.436,2993.632


In [73]:
# See how big the data
print(zhvi.shape) # (3071, 329)
print(zori.shape) # (1407, 149)

(3071, 329)
(1407, 149)


In [74]:
# Descriptive Statistics
zhvi.describe()

,RegionID,SizeRank,StateCodeFIPS,MunicipalCodeFIPS,2000-01-31,2000-02-29,2000-03-31,2000-04-30,2000-05-31,2000-06-30,...,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31,2026-04-30,2026-05-31,2026-06-30,2026-07-31,2026-08-31
count,3071.000,3071.000,3071.000,3071.000,1034.000,1036.000,1038.000,1040.000,1043.000,1045.000,...,3071.000,3071.000,3071.000,3071.000,3071.000,3071.000,3071.000,3071.000,3071.000,3071.000
mean,1900.211,1579.919,30.279,103.372,112864.576,113073.336,113232.751,113850.409,114498.674,115137.397,...,268342.235,269364.586,270511.812,271771.860,272902.610,273480.109,273547.150,273508.100,273645.696,274047.525
std,12612.429,921.396,15.069,107.814,57583.538,57756.302,57999.689,58621.819,59262.559,59955.393,...,169755.103,170549.218,171231.270,171755.534,172089.668,172016.194,171936.474,171966.893,172508.989,173369.904
min,66.000,0.000,1.000,1.000,27426.277,27493.783,27580.213,27765.629,27999.460,28240.602,...,45867.349,43787.711,42291.769,41775.147,42272.404,42893.045,43174.010,43465.974,43156.222,42865.832
25%,873.500,776.500,18.500,35.000,75047.650,75252.063,75192.062,75584.974,75912.699,76329.916,...,167053.491,168115.491,169230.919,170559.607,171949.449,172576.480,172669.611,172488.468,172506.632,173139.407
50%,1670.000,1580.000,29.000,79.000,99177.614,99400.030,99650.828,99936.442,100392.659,100749.938,...,228194.008,229398.408,230576.732,232156.724,233469.221,234610.950,234578.152,234466.602,234593.053,235072.481
75%,2477.500,2377.500,45.000,133.000,134096.720,134104.795,134227.194,134872.066,135930.532,136439.420,...,321285.752,322241.392,322858.895,324072.278,325081.413,325271.505,325426.376,325183.047,325272.399,325916.139
max,698720.000,3213.000,56.000,840.000,665445.104,667606.628,670799.533,677865.370,687139.327,696466.426,...,2966699.358,2997940.936,3027361.613,3056786.773,3082305.516,3096086.577,3105437.807,3118158.553,3140813.292,3171305.345


In [75]:
# prints information about a DataFrame
print(zhvi.info())
print(zori.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3071 entries, 0 to 3070
Columns: 329 entries, RegionID to 2026-08-31
dtypes: float64(320), int64(4), object(5)
memory usage: 7.7+ MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1407 entries, 0 to 1406
Columns: 149 entries, RegionID to 2026-08-31
dtypes: float64(140), int64(4), object(5)
memory usage: 1.6+ MB
None


In [76]:
# Missing values
print(zhvi.isna().sum())
print(zori.isna().sum())

RegionID      0
SizeRank      0
RegionName    0
RegionType    0
StateName     0
             ..
2026-04-30    0
2026-05-31    0
2026-06-30    0
2026-07-31    0
2026-08-31    0
Length: 329, dtype: int64
RegionID        0
SizeRank        0
RegionName      0
RegionType      0
StateName       0
             ... 
2026-04-30    224
2026-05-31    196
2026-06-30    169
2026-07-31    114
2026-08-31      1
Length: 149, dtype: int64


In [77]:
# Check Duplicates
print(zhvi.duplicated().sum())
print(zori.duplicated().sum())

0
0


#### Make sure the data match across all three datasets, so I filtered the dates to include only records from 2020 to 2025.

In [78]:
# # ZHVI
# columns = ["RegionID", "SizeRank", "RegionName", "RegionType", "StateName"]

# filter_20 = columns + [i for i in zhvi.columns if i[0].isdigit() and i >= "2020-01"]

# zhvi_20 = zhvi[filter_20]

# zhvi_20

In [79]:
# # ZORI
# columns = ["RegionID", "SizeRank", "RegionName", "RegionType", "StateName"]

# filter_20 = columns + [i for i in zori.columns if i[0].isdigit() and i >= "2020-01"]

# zori_20 = zori[filter_20]

# zori_20

#### Write a function for above code, so that only need to call once.

In [80]:
# I'm trying to write above code into a function, which makes it clean.

def cleaning_data(data, start = "2020-01", end = "2026-01"):

  columns = ["RegionID", "SizeRank", "RegionName", "RegionType", "StateName", "StateCodeFIPS", "MunicipalCodeFIPS"]

  filter_20 = columns + [i for i in data.columns if i[0].isdigit() and i >= start and i < end]

  return data[filter_20]

zhvi, zori = (cleaning_data(d) for d in (zhvi, zori))

# ProblemSet 1

### California Counties Level




### ZHVI Data

In [81]:
# Make a copy so that don't alter the original data
zhvi_CA = zhvi.copy()

In [82]:
zhvi_CA = zhvi_CA[zhvi_CA["StateName"] == "CA"]

#### Clean the data

In [83]:
# See the data types in the dataframe
zhvi_CA.dtypes

,0
RegionID,int64
SizeRank,int64
RegionName,object
RegionType,object
StateName,object
...,...
2025-08-31,float64
2025-09-30,float64
2025-10-31,float64
2025-11-30,float64


In [84]:
zhvi_CA.head()

,RegionID,SizeRank,RegionName,RegionType,StateName,StateCodeFIPS,MunicipalCodeFIPS,2020-01-31,2020-02-29,2020-03-31,...,2025-03-31,2025-04-30,2025-05-31,2025-06-30,2025-07-31,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31
0,3101,0,Los Angeles County,county,CA,6,37,646052.857,647272.503,647222.143,...,884866.939,879512.499,874473.519,870275.199,868351.199,868315.469,870417.973,873252.450,876483.587,880007.554
4,2841,4,San Diego County,county,CA,6,73,606555.564,610767.028,615376.796,...,947506.348,944322.543,939461.348,934181.794,929769.210,926319.844,924448.871,924417.075,925424.341,927598.249
5,1286,5,Orange County,county,CA,6,59,723754.034,725642.229,727855.268,...,1166614.417,1164681.420,1160046.239,1154651.883,1150085.574,1146465.957,1145150.012,1146832.877,1150833.716,1156617.195
9,2832,9,Riverside County,county,CA,6,65,402042.996,403975.716,406194.046,...,616812.043,614905.468,612533.637,610141.022,607211.107,604404.772,602738.043,602464.026,603462.293,605008.058
13,3250,13,San Bernardino County,county,CA,6,71,369240.934,370703.710,371159.780,...,557034.269,554614.849,552003.341,549477.732,547314.818,545565.572,544587.410,544591.018,545717.821,547407.809


In [85]:
## Reshape the Data https://pandas.pydata.org/docs/reference/api/pandas.melt.html
## The data is often provided in a wide format, so I will unpivot it from wide to long format.

id_col = ["RegionID", "SizeRank", "RegionName", "RegionType", "StateName", "StateCodeFIPS", "MunicipalCodeFIPS"]

date_col = []
for col in zhvi_CA.columns:
  if col not in id_col:
    date_col.append(col)

zhvi_CA = zhvi_CA.melt(
    id_vars = id_col,
    value_vars = date_col,
    var_name = "Date",
    value_name = "ZHVI"
)

In [86]:
# Drop unused columns
zhvi_CA = zhvi_CA.drop(columns=["RegionID", "SizeRank", "RegionType", "StateName"], axis=1)

In [87]:
zhvi_CA

,RegionName,StateCodeFIPS,MunicipalCodeFIPS,Date,ZHVI
0,Los Angeles County,6,37,2020-01-31,646052.857
1,San Diego County,6,73,2020-01-31,606555.564
2,Orange County,6,59,2020-01-31,723754.034
3,Riverside County,6,65,2020-01-31,402042.996
4,San Bernardino County,6,71,2020-01-31,369240.934
...,...,...,...,...,...
4171,Trinity County,6,105,2025-12-31,261318.979
4172,Mono County,6,51,2025-12-31,751906.256
4173,Modoc County,6,49,2025-12-31,189287.471
4174,Sierra County,6,91,2025-12-31,329869.433


In [88]:
zhvi_CA.dtypes

,0
RegionName,object
StateCodeFIPS,int64
MunicipalCodeFIPS,int64
Date,object
ZHVI,float64


In [89]:
zhvi_CA.select_dtypes(include='float')

,ZHVI
0,646052.857
1,606555.564
2,723754.034
3,402042.996
4,369240.934
...,...
4171,261318.979
4172,751906.256
4173,189287.471
4174,329869.433


In [90]:
# Convert Dates to datetime type
zhvi_CA["Date"] = pd.to_datetime(zhvi_CA["Date"], errors = "coerce") # Turns invalid dates into NaT(Not a Time)

# remove the space left&right
zhvi_CA["RegionName"] = zhvi_CA["RegionName"].str.strip()

# Check missing value in Date column
zhvi_CA["Date"].isna().sum()

np.int64(0)

In [91]:
# sort the data
zhvi_CA = zhvi_CA.sort_values(
    by = ["RegionName", "Date"]).reset_index(drop=True)

zhvi_CA.head()

,RegionName,StateCodeFIPS,MunicipalCodeFIPS,Date,ZHVI
0,Alameda County,6,1,2020-01-31,842261.950
1,Alameda County,6,1,2020-02-29,845568.279
2,Alameda County,6,1,2020-03-31,849961.749
3,Alameda County,6,1,2020-04-30,854051.748
4,Alameda County,6,1,2020-05-31,853204.560


### ZORI Data

The index is dollar-denominated by computing the mean of listed rents that fall into the 35th to 65th percentile range for all homes and apartments in a given region, which is weighted to reflect the rental housing stock.

In [92]:
# Make a copy so that don't alter the original data
zori_CA = zori.copy()

In [93]:
zori_CA = zori_CA[zori_CA["StateName"] == "CA"]

#### Clean the data

In [94]:
# See the data types in the dataframe
zori_CA.dtypes

,0
RegionID,int64
SizeRank,int64
RegionName,object
RegionType,object
StateName,object
...,...
2025-08-31,float64
2025-09-30,float64
2025-10-31,float64
2025-11-30,float64


In [95]:
## Reshape the Data
## The data is often provided in a wide format, so I will unpivot it from wide to long format.

id_col = ["RegionID", "SizeRank", "RegionName", "RegionType", "StateName", "StateCodeFIPS", "MunicipalCodeFIPS"]

date_columns = [
    col for col in zori_CA.columns
    if col not in id_col
]

zori_CA = zori_CA.melt(
    id_vars = id_col,
    value_vars = date_columns,
    var_name = "Date",
    value_name = "ZORI"
)

In [96]:
# Drop unused columns
zori_CA = zori_CA.drop(columns=["RegionID", "SizeRank", "RegionType", "StateName"], axis=1)

In [97]:
zori_CA.dtypes

,0
RegionName,object
StateCodeFIPS,int64
MunicipalCodeFIPS,int64
Date,object
ZORI,float64


In [98]:
zori_CA.select_dtypes(include='float')

,ZORI
0,2233.546
1,2082.318
2,2270.439
3,1740.369
4,1648.598
...,...
3523,1628.664
3524,1852.713
3525,NaN
3526,NaN


In [99]:
# Convert Dates to datetime type
zori_CA["Date"] = pd.to_datetime(
    zori_CA["Date"],
    errors="coerce" # Turns invalid dates into NaT(Not a Time)
)

In [100]:
# sort the data
zori_CA = zori_CA.sort_values(
    by = ["RegionName", "Date"]).reset_index(drop=True)

zori_CA.head()

,RegionName,StateCodeFIPS,MunicipalCodeFIPS,Date,ZORI
0,Alameda County,6,1,2020-01-31,2547.423
1,Alameda County,6,1,2020-02-29,2548.081
2,Alameda County,6,1,2020-03-31,2563.948
3,Alameda County,6,1,2020-04-30,2561.371
4,Alameda County,6,1,2020-05-31,2556.497


#### Merge the ZORI & ZHVI dataframe

In [101]:
zhvi_CA["FIPS"] = (zhvi_CA["StateCodeFIPS"].astype(str).str.zfill(2)
                   + zhvi_CA["MunicipalCodeFIPS"].astype(str).str.zfill(3))

zori_CA["FIPS"] = (zori_CA["StateCodeFIPS"].astype(str).str.zfill(2)
                   + zori_CA["MunicipalCodeFIPS"].astype(str).str.zfill(3))

In [102]:
zori_CA.drop(columns=["StateCodeFIPS", "MunicipalCodeFIPS"], inplace=True)
zhvi_CA.drop(columns=["StateCodeFIPS", "MunicipalCodeFIPS"], inplace=True)

In [103]:
housing = zhvi_CA.merge(
    zori_CA,
    on=["FIPS"],
    how="outer",
    indicator = True
)
housing[housing["_merge"] == "left_only"]

,RegionName_x,Date_x,ZHVI,FIPS,RegionName_y,Date_y,ZORI,_merge
5184,Alpine County,2020-01-31,389374.972,06003,NaN,NaT,NaN,left_only
5185,Alpine County,2020-02-29,389221.795,06003,NaN,NaT,NaN,left_only
5186,Alpine County,2020-03-31,388438.388,06003,NaN,NaT,NaN,left_only
5187,Alpine County,2020-04-30,388918.625,06003,NaN,NaT,NaN,left_only
5188,Alpine County,2020-05-31,389024.193,06003,NaN,NaT,NaN,left_only
...,...,...,...,...,...,...,...,...
228739,Trinity County,2025-08-31,263132.164,06105,NaN,NaT,NaN,left_only
228740,Trinity County,2025-09-30,262568.495,06105,NaN,NaT,NaN,left_only
228741,Trinity County,2025-10-31,261993.268,06105,NaN,NaT,NaN,left_only
228742,Trinity County,2025-11-30,261540.047,06105,NaN,NaT,NaN,left_only


In [104]:
# No rent counties
no_rent = housing.loc[housing["_merge"] == "left_only", "RegionName_x"].unique()
print(f"{len(no_rent)} counties have no ZORI at all:\n", sorted(no_rent))

9 counties have no ZORI at all:
 ['Alpine County', 'Colusa County', 'Lassen County', 'Mariposa County', 'Modoc County', 'Mono County', 'Plumas County', 'Sierra County', 'Trinity County']


### Wikipedia List of counties in California data

Source: https://en.wikipedia.org/wiki/List_of_counties_in_California

#### Get the data from Web

In [105]:
# import requests

# url = "https://en.wikipedia.org/wiki/List_of_counties_in_California"

# header = {
#   "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/152.0.0.0 Safari/537.36",
#   "X-Requested-With": "XMLHttpRequest"
# }

# r = requests.get(url, headers=header, timeout = 10)

# # print(r)
# population = pd.read_html(r.text, match='County') # scrape all HTML tables (the <table> tag) from webpage

# Select the first DataFrame from the list and save it to CSV
# population[0].to_csv("population_CA.csv", index=False)

In [106]:
population = pd.read_csv("https://github.com/HowardChen0211/dataManPy/raw/refs/heads/main/data/population_CA.csv")

In [107]:
population.head()

,County,FIPS code[8],County seat[9],Est.[9],Formed from,Etymology[10],General Law or Charter [6],Population (2025)[11],Area[9],Map
0,Alameda County,1,Oakland,1853,Contra Costa and Santa Clara,"The oak and other trees, once abundant in the ...",Charter,1636630,"738 sq mi (1,911 km2)",NaN
1,Alpine County,3,Markleeville,1864,"Amador, El Dorado, Calaveras, Mono and Tuolumne",Location high in the Sierra Nevada; alpine ref...,General Law,1043,"739 sq mi (1,914 km2)",NaN
2,Amador County,5,Jackson,1854,Calaveras,"Jose Maria Amador (1794–1883), a soldier, ranc...",General Law,41876,"593 sq mi (1,536 km2)",NaN
3,Butte County,7,Oroville,1850,original,"Sutter Buttes, which were mistakenly thought t...",Charter,209211,"1,640 sq mi (4,248 km2)",NaN
4,Calaveras County,9,San Andreas,1850,original,"Calaveras River; calaveras is Spanish for ""sku...",General Law,46605,"1,020 sq mi (2,642 km2)",NaN


#### Clean the data

In [108]:
population = population[["County", "FIPS code[8]", "Population (2025)[11]"]]

In [109]:
population.rename(columns={
    "FIPS code[8]": "FIPS",
    "Population (2025)[11]": "pop_2025"}, inplace = True)

In [110]:
population.dtypes

,0
County,object
FIPS,int64
pop_2025,int64


In [111]:
population["FIPS"] = "06" + population["FIPS"].astype(str).str.zfill(3)

In [112]:
population

,County,FIPS,pop_2025
0,Alameda County,06001,1636630
1,Alpine County,06003,1043
2,Amador County,06005,41876
3,Butte County,06007,209211
4,Calaveras County,06009,46605
5,Colusa County,06011,21836
6,Contra Costa County,06013,1170070
7,Del Norte County,06015,26410
8,El Dorado County,06017,192323
9,Fresno County,06019,1035456


### California Unemployment Data - 2024
Source: Social explorer
American Community Survey 2020--2024 (5-Year-Estimates) https://www.socialexplorer.com/reports/socialexplorer/en/report/bac15980-b308-11f1-9512-c38da70940e6

In [113]:
# Labor Force 16 Years and Over
unemployment = pd.read_csv("https://github.com/HowardChen0211/gisPy/raw/refs/heads/main/CA_UnemploymentRate.csv")
unemployment.head()

,FIPS,GeoLevel,NAME,Qualified Area Name,Area (Land),Area (Water),Civilian Population in Labor Force 16 Years and Over,Civilian Population in Labor Force 16 Years and Over: Employed,Civilian Population in Labor Force 16 Years and Over: Unemployed
0,Geo_FIPS,_geo_level_,Geo_NAME,Geo_qname,Geo_AREALAND,Geo_AREAWATER,SE_A17005_001,SE_A17005_002,SE_A17005_003
1,06001,SL050,"Alameda County, California","Alameda County, California",1910010500,216909647,908995,860129,48866
2,06003,SL050,"Alpine County, California","Alpine County, California",1912292607,12557304,837,800,37
3,06005,SL050,"Amador County, California","Amador County, California",1539967082,29437117,17257,16325,932
4,06007,SL050,"Butte County, California","Butte County, California",4238545149,105204062,100743,93228,7515


#### Clean the data

In [114]:
unemployment.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59 entries, 0 to 58
Data columns (total 9 columns):
 #   Column                                                            Non-Null Count  Dtype 
---  ------                                                            --------------  ----- 
 0   FIPS                                                              59 non-null     object
 1   GeoLevel                                                          59 non-null     object
 2   NAME                                                              59 non-null     object
 3   Qualified Area Name                                               59 non-null     object
 4   Area (Land)                                                       59 non-null     object
 5   Area (Water)                                                      59 non-null     object
 6   Civilian Population in Labor Force 16 Years and Over              59 non-null     object
 7   Civilian Population in Labor Force 16 Years and

In [115]:
unemployment.isna().sum()

,0
FIPS,0
GeoLevel,0
NAME,0
Qualified Area Name,0
Area (Land),0
Area (Water),0
Civilian Population in Labor Force 16 Years and Over,0
Civilian Population in Labor Force 16 Years and Over: Employed,0
Civilian Population in Labor Force 16 Years and Over: Unemployed,0


In [116]:
unemployment.duplicated().sum()

np.int64(0)

In [117]:
# Choose the columns
unemployment = unemployment[["FIPS",
                             "NAME",
                             'Civilian Population in Labor Force 16 Years and Over',
                             'Civilian Population in Labor Force 16 Years and Over: Employed',
                             'Civilian Population in Labor Force 16 Years and Over: Unemployed']].iloc[1:, :]

# Rename the columns
unemployment.rename(columns={"NAME":"County",
                             "Civilian Population in Labor Force 16 Years and Over": "Labor Force",
                             "Civilian Population in Labor Force 16 Years and Over: Employed": "Employed",
                             'Civilian Population in Labor Force 16 Years and Over: Unemployed':"Unemployed"}, inplace=True)

In [118]:
# get rid of ' County, California'
unemployment["County"] = unemployment["County"].str.replace(", California", "")
unemployment.head()

,FIPS,County,Labor Force,Employed,Unemployed
1,06001,Alameda County,908995,860129,48866
2,06003,Alpine County,837,800,37
3,06005,Amador County,17257,16325,932
4,06007,Butte County,100743,93228,7515
5,06009,Calaveras County,19422,18545,877


In [119]:
# Convert the data type
unemployment['Labor Force'] = unemployment['Labor Force'].astype(int)
unemployment['Employed'] = unemployment['Employed'].astype(int)
unemployment['Unemployed'] = unemployment['Unemployed'].astype(int)
unemployment['Rate'] = unemployment['Unemployed'] / unemployment['Labor Force'] * 100

unemployment.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58 entries, 1 to 58
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   FIPS         58 non-null     object 
 1   County       58 non-null     object 
 2   Labor Force  58 non-null     int64  
 3   Employed     58 non-null     int64  
 4   Unemployed   58 non-null     int64  
 5   Rate         58 non-null     float64
dtypes: float64(1), int64(3), object(2)
memory usage: 2.8+ KB


In [120]:
unemployment.head()

,FIPS,County,Labor Force,Employed,Unemployed,Rate
1,06001,Alameda County,908995,860129,48866,5.376
2,06003,Alpine County,837,800,37,4.421
3,06005,Amador County,17257,16325,932,5.401
4,06007,Butte County,100743,93228,7515,7.460
5,06009,Calaveras County,19422,18545,877,4.515


### Merge the unemp & population dataframe

In [121]:
# It's good.
# pop_unemp = pd.merge(unemployment, population, on="FIPS", how="outer", indicator=True)
# pop_unemp.query("_merge == 'left_only'")

In [122]:
pop_unemp = pd.merge(unemployment, population.drop(columns=["County"]), on="FIPS", how="inner")
pop_unemp.head()

,FIPS,County,Labor Force,Employed,Unemployed,Rate,pop_2025
0,06001,Alameda County,908995,860129,48866,5.376,1636630
1,06003,Alpine County,837,800,37,4.421,1043
2,06005,Amador County,17257,16325,932,5.401,41876
3,06007,Butte County,100743,93228,7515,7.460,209211
4,06009,Calaveras County,19422,18545,877,4.515,46605


In [125]:
pop_unemp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   FIPS         58 non-null     object 
 1   County       58 non-null     object 
 2   Labor Force  58 non-null     int64  
 3   Employed     58 non-null     int64  
 4   Unemployed   58 non-null     int64  
 5   Rate         58 non-null     float64
 6   pop_2025     58 non-null     int64  
dtypes: float64(1), int64(4), object(2)
memory usage: 3.3+ KB


In [126]:
# No data in ZORI
counties_no_zori = ['Alpine County', 'Colusa County', 'Lassen County',
            'Mariposa County', 'Modoc County', 'Mono County',
            'Plumas County', 'Sierra County', 'Trinity County']


has_zori = pop_unemp[~pop_unemp["County"].isin(counties_no_zori)]["County"].unique()

pop_unemp["has_zori"] = pop_unemp["County"].isin(has_zori) # True/False

pop_unemp.groupby("has_zori").agg(
    n_counties = ("County", "count"),
    total_pop = ("pop_2025", "sum"),
    median_pop = ("pop_2025", "median"),
    median_unemp = ("Rate", "median"),
)

,n_counties,total_pop,median_pop,median_unemp
has_zori,,,,
False,9,126246,15720.000,8.320
True,49,39229063,258852.000,6.418


In counties lacking rental data, the labor market is also relatively weak.

## Reference

1. Zillow Housing: https://www.zillow.com/research/data/

2. Source: U.S. Census Bureau, Population Division ;
Annual Estimates of the Resident Population for Metropolitan Statistical Areas in the United States and Puerto Rico: April 1, 2020 to July 1, 2025 (CBSA-MET-EST2025-POP)
